# Spanish↔English MT with embedding + uniformity regularization

This notebook keeps the training loop explicit:
- no `Seq2SeqTrainingArguments`
- no custom `Trainer` subclass
- plain `model.train()`, `loss.backward()`, `optimizer.step()`

`MODEL_NAME` below is set to **Helsinki-NLP/opus-mt-es-en** for Spanish → English.
For English → Spanish, switch to **Helsinki-NLP/opus-mt-en-es** and swap `SRC_LANG` / `TGT_LANG`.


In [13]:
print('test')

test


In [14]:
import os
import math
import json
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

import pandas as pd

In [16]:
# Optional metrics
try:
    from comet import download_model, load_from_checkpoint
    COMET_AVAILABLE = True
except Exception:
    COMET_AVAILABLE = False

try:
    from bert_score import score as bertscore_score
    BERTSCORE_AVAILABLE = True
except Exception:
    BERTSCORE_AVAILABLE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


In [17]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.11.0+cu128
CUDA Available: True


In [18]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version PyTorch Wants:", torch.version.cuda)
print("Device Count:", torch.cuda.device_count())

PyTorch Version: 2.11.0+cu128
CUDA Available: True
CUDA Version PyTorch Wants: 12.8
Device Count: 1


In [19]:
# -------------------------
# Config
# -------------------------
MODEL_NAME = "Helsinki-NLP/opus-mt-es-en"
SRC_LANG = "es"
TGT_LANG = "en"

DATASET_NAME = "Helsinki-NLP/opus_books"
DATASET_CONFIG = "en-es"

MAX_SRC_LEN = 128
MAX_TGT_LEN = 128
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR = 5e-5
GRAD_CLIP = 1.0

# Regularization toggles
LAMBDA_EMBED = 0.1
GAMMA_UNIFORM = 0.01
EMBED_LOSS_TYPE = "cosine"   # "cosine" or "l2"
USE_WORD_AWARE_LOSS = True    # set False to use token-level embedding loss
USE_UNIFORM_LOSS = True       # set False to disable Wang & Isola uniformity term
EMBEDDING_SOURCE = "input"    # "input", "output", or "shared"
UNIFORMITY_SAMPLE_SIZE = 256   # keep L_uniform cheap

# Evaluation / logging
EVAL_GENERATION_MAX_BATCHES = 4   # cap expensive generation eval
EVAL_GENERATION_EVERY = 1         # set >1 to do generation less often
LOG_DIR = Path("logs")
RUN_NAME = "opus_mt_es_en_embed_reg"

In [20]:
# -------------------------
# Data
# -------------------------
def load_translation_data(seed=42, test_size=0.1, val_size=0.1):
    # default: 10% test, 10% val, 80% train
    ds = load_dataset(DATASET_NAME, DATASET_CONFIG)

    # opus_books only has a train split, so we make val/test ourselves
    tmp = ds["train"].train_test_split(test_size=(test_size + val_size), seed=seed)
    train_hf = tmp["train"]

    tmp2 = tmp["test"].train_test_split(
        test_size=val_size / (test_size + val_size),
        seed=seed,
    )
    val_hf = tmp2["train"]
    test_hf = tmp2["test"]
    return train_hf, val_hf, test_hf


def sentencepiece_word_spans(tokenizer, token_ids):
    """
    Group SentencePiece/subword tokens into word-like spans using the leading ▁ marker.
    Returns spans as lists of positions into the original token sequence.
    """
    special_ids = set(tokenizer.all_special_ids)
    token_ids = [int(t) for t in token_ids]

    valid_positions = [
        i for i, tid in enumerate(token_ids)
        if tid != -100 and tid not in special_ids
    ]
    if not valid_positions:
        return []

    valid_tokens = tokenizer.convert_ids_to_tokens([token_ids[i] for i in valid_positions])

    spans_local = []
    current = [0]
    for local_idx, tok in enumerate(valid_tokens[1:], start=1):
        if tok.startswith("▁"):
            spans_local.append(current)
            current = [local_idx]
        else:
            current.append(local_idx)
    spans_local.append(current)

    # map local indices back to original sequence positions
    return [[valid_positions[j] for j in span] for span in spans_local]


class ParallelTextDataset(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]["translation"]
        return {"src": ex[SRC_LANG], "tgt": ex[TGT_LANG]}

# a collate fn that tokenizes and pads a batch of examples, and also precomputes word spans for the word-aware loss if needed
def make_collate_fn(tokenizer, max_src_len=MAX_SRC_LEN, max_tgt_len=MAX_TGT_LEN, use_word_aware_loss=False):
    def collate_fn(batch):
        src_texts = [x["src"] for x in batch]
        tgt_texts = [x["tgt"] for x in batch]

        enc = tokenizer(
            src_texts,
            max_length=max_src_len,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        tgt = tokenizer(
            text_target=tgt_texts,
            max_length=max_tgt_len,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        # For seq2seq models in the HuggingFace library, the labels are just the tokenized target sequences, 
        # but with padding tokens replaced by -100 so they don't contribute to the loss.
        labels = tgt["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100

        enc["labels"] = labels
        enc["src_text"] = src_texts
        enc["tgt_text"] = tgt_texts

        # Precompute spans once per batch so the word-aware loss doesn't redo this work.
        if use_word_aware_loss:
            enc["word_spans"] = [
                sentencepiece_word_spans(tokenizer, row.tolist())
                for row in labels
            ]

        return enc

    return collate_fn


def move_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out

In [21]:
# -------------------------
# Losses
# -------------------------
def get_embedding_weight(model, embedding_source="input"):
    """
    Choose which embedding matrix to use for the auxiliary loss.
    """
    if embedding_source == "input":
        emb = model.get_input_embeddings()
    elif embedding_source == "output":
        emb = model.get_output_embeddings()
        if emb is None:
            emb = model.get_input_embeddings()
    elif embedding_source == "shared":
        emb = None
        if hasattr(model, "model") and hasattr(model.model, "shared"):
            emb = model.model.shared
        elif hasattr(model, "shared"):
            emb = model.shared
        if emb is None:
            emb = model.get_input_embeddings()
    else:
        raise ValueError(f"Unknown embedding_source: {embedding_source}")

    return emb.weight


def embedding_loss_from_logits(
    logits,
    labels,
    token_embedding_weight,
    tokenizer,
    pad_token_id,
    loss_type="cosine",
    use_word_aware_loss=False,
    word_spans=None,
):
    """
    If use_word_aware_loss=False:
        compare per-token predicted embeddings against per-token target embeddings.

    If use_word_aware_loss=True:
        group SentencePiece subwords into word-like spans, average subword embeddings
        within each span, and compare word-level predicted and target embeddings.
    """
    probs = torch.softmax(logits, dim=-1)   # [B, T, V]
    e_hat = probs @ token_embedding_weight  # [B, T, H]

    if not use_word_aware_loss:
        gt_ids = labels.masked_fill(labels == -100, pad_token_id)
        e_gt = token_embedding_weight[gt_ids]      # [B, T, H]
        valid = labels != -100

        if valid.sum() == 0:
            return logits.new_tensor(0.0), e_hat, None

        if loss_type == "l2":
            per_item = (e_hat - e_gt).pow(2).sum(dim=-1)
        elif loss_type == "cosine":
            per_item = 1.0 - F.cosine_similarity(e_hat, e_gt, dim=-1)
        else:
            raise ValueError(f"Unknown loss_type: {loss_type}")

        return per_item[valid].mean(), e_hat, e_hat[valid]
    
    else:
        # Word-aware branch
        if word_spans is None:
            word_spans = [
                sentencepiece_word_spans(tokenizer, row.tolist())
                for row in labels
            ]

        word_pred_vecs = []
        word_gt_vecs = []

        for b, spans in enumerate(word_spans):
            if not spans:
                continue

            for span in spans:
                idx = torch.as_tensor(span, device=logits.device, dtype=torch.long)
                valid = labels[b, idx] != -100
                if valid.sum() == 0:
                    continue

                idx = idx[valid]
                pred_vec = e_hat[b, idx].mean(dim=0)
                gt_vec = token_embedding_weight[labels[b, idx]].mean(dim=0)

                word_pred_vecs.append(pred_vec)
                word_gt_vecs.append(gt_vec)

        if len(word_pred_vecs) == 0:
            return logits.new_tensor(0.0), e_hat, None

        word_pred_vecs = torch.stack(word_pred_vecs, dim=0)
        word_gt_vecs = torch.stack(word_gt_vecs, dim=0)

        if loss_type == "l2":
            per_item = (word_pred_vecs - word_gt_vecs).pow(2).sum(dim=-1)
        elif loss_type == "cosine":
            per_item = 1.0 - F.cosine_similarity(word_pred_vecs, word_gt_vecs, dim=-1)
        else:
            raise ValueError(f"Unknown loss_type: {loss_type}")

        return per_item.mean(), e_hat, word_pred_vecs


def uniformity_loss(embeddings, enabled=True, sample_size=UNIFORMITY_SAMPLE_SIZE):
    """
    Wang & Isola-style uniformity loss:
        log mean_{i != j} exp(-2 ||z_i - z_j||^2)
    """
    if not enabled or embeddings is None or embeddings.size(0) < 2:
        if embeddings is None:
            return torch.tensor(0.0, device=DEVICE)
        return embeddings.new_tensor(0.0)

    z = embeddings
    if z.size(0) > sample_size:
        idx = torch.randperm(z.size(0), device=z.device)[:sample_size]
        z = z[idx]

    z = F.normalize(z, dim=-1)
    dist_sq = torch.cdist(z, z).pow(2)

    i, j = torch.triu_indices(z.size(0), z.size(0), offset=1, device=z.device)
    pair_terms = torch.exp(-2.0 * dist_sq[i, j])
    return torch.log(pair_terms.mean().clamp_min(1e-12))


def total_loss(
    logits,
    labels,
    token_embedding_weight,
    tokenizer,
    pad_token_id,
    lambda_embed=LAMBDA_EMBED,
    gamma_uniform=GAMMA_UNIFORM,
    embed_loss_type=EMBED_LOSS_TYPE,
    use_word_aware_loss=USE_WORD_AWARE_LOSS,
    use_uniform_loss=USE_UNIFORM_LOSS,
    word_spans=None,
):
    ce_loss = F.cross_entropy(
        # HuggingFace seq2seq models expect labels to be in the shape [B, T] with -100 for padding, and they compute the 
        # loss internally by reshaping to [B*T, V] and [B*T], so we do the same here for our auxiliary loss.
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100,
    )

    embed_loss, e_hat, aux_vecs = embedding_loss_from_logits(
        logits=logits,
        labels=labels,
        token_embedding_weight=token_embedding_weight,
        tokenizer=tokenizer,
        pad_token_id=pad_token_id,
        loss_type=embed_loss_type,
        use_word_aware_loss=use_word_aware_loss,
        word_spans=word_spans,
    )
    # if use_uniform_loss == False, this will just return 0.0
    uni_loss = uniformity_loss(aux_vecs, enabled=use_uniform_loss)

    loss = ce_loss + lambda_embed * embed_loss + gamma_uniform * uni_loss
    stats = {
        "loss": float(loss.detach().cpu()),
        "ce_loss": float(ce_loss.detach().cpu()),
        "embed_loss": float(embed_loss.detach().cpu()),
        "uniform_loss": float(uni_loss.detach().cpu()),
        "n_aux_vecs": int(0 if aux_vecs is None else aux_vecs.size(0)),
    }
    return loss, stats

In [ ]:
# -------------------------
# Metrics + evaluation
# -------------------------
def decode_texts(tokenizer, ids):
    return tokenizer.batch_decode(ids, skip_special_tokens=True)


def _aggregate_stats(sum_stats, count):
    # Avoid division by zero if count is 0 (e.g. no valid tokens or no aux vecs in the batch)
    if count == 0:
        return {k: 0.0 for k in sum_stats}
    # We multiplied by batch size when accumulating, so now we divide by total count of items to get the average.
    return {k: v / count for k, v in sum_stats.items()}


def append_jsonl(path, record):
    # Append a single JSON-serializable record to a JSONL file, creating parent directories if needed.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


@torch.no_grad()
def evaluate_loss(
    model,
    dataloader,
    tokenizer,
    lambda_embed=LAMBDA_EMBED,
    gamma_uniform=GAMMA_UNIFORM,
    embed_loss_type=EMBED_LOSS_TYPE,
    use_word_aware_loss=USE_WORD_AWARE_LOSS,
    use_uniform_loss=USE_UNIFORM_LOSS,
    embedding_source=EMBEDDING_SOURCE,
):
    """
    Fast evaluation: teacher-forced forward pass only.
    Much faster than generation-based evaluation.
    """
    model.eval()

    sum_stats = defaultdict(float)
    n_examples = 0

    for batch in tqdm(dataloader, desc="Eval loss"):
        batch = move_to_device(batch, DEVICE)

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        logits = outputs.logits

        loss, stats = total_loss(
            logits=logits,
            labels=batch["labels"],
            token_embedding_weight=get_embedding_weight(model, embedding_source=embedding_source),
            tokenizer=tokenizer,
            pad_token_id=tokenizer.pad_token_id,
            lambda_embed=lambda_embed,
            gamma_uniform=gamma_uniform,
            embed_loss_type=embed_loss_type,
            use_word_aware_loss=use_word_aware_loss,
            use_uniform_loss=use_uniform_loss,
            word_spans=batch.get("word_spans"),
        )

        bs = batch["labels"].size(0)
        n_examples += bs
        for k, v in stats.items():
            sum_stats[k] += float(v) * bs

    return _aggregate_stats(sum_stats, n_examples)


@torch.no_grad()
def evaluate_generation(
    model,
    dataloader,
    tokenizer,
    use_bertscore=False,
    comet_model=None,
    max_batches=None,
    save_scores_path=None,
):
    """
    Expensive evaluation: autoregressive generation + overlap/semantic metrics.
    Limit max_batches to keep this manageable on one GPU.
    
    If save_scores_path is provided, saves a CSV with per-example scores (source, reference, hypothesis, bertscore, comet).
    Returns: metrics, sources, refs, hyps, results_df
    """
    bleu_metric = BLEU()
    chrf_metric = CHRF(word_order=2)  # chrF++

    # we enable use_cache for faster generation, but we want to restore the original setting afterwards in case the caller relies on it for something else
    prev_use_cache = model.config.use_cache
    model.config.use_cache = True
    model.eval()

    sources, refs, hyps = [], [], []

    # we generate with batch_size=1 for simplicity, since generation can produce variable-length outputs that are tricky to collate
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Eval gen")):
        if max_batches is not None and batch_idx >= max_batches:
            break

        sources.extend(batch["src_text"])
        refs.extend(batch["tgt_text"])

        batch = move_to_device(batch, DEVICE)
        gen_ids = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_new_tokens=MAX_TGT_LEN,
            num_beams=1,
            do_sample=False,
        )
        # gen_ids is a list of token ID sequences, one per example in the batch. We decode them to get the generated text hypotheses.
        hyps.extend(decode_texts(tokenizer, gen_ids))

    metrics = {
        "bleu": bleu_metric.corpus_score(hyps, [refs]).score if hyps else 0.0,
        "chrfpp": chrf_metric.corpus_score(hyps, [refs]).score if hyps else 0.0,
        "n_eval_examples": len(hyps),
    }

    # Build dataframe to track per-example scores
    results_data = {
        "source": sources,
        "reference": refs,
        "hypothesis": hyps,
    }

    # BERTScore can be very slow, so we make it optional and also skip it if there are no hypotheses (e.g. if max_batches=0)
    if use_bertscore and BERTSCORE_AVAILABLE and hyps:
        P, R, F1 = bertscore_score(hyps, refs, lang="en", rescale_with_baseline=True)
        bertscore_f1_list = F1.tolist()  # convert to list for dataframe
        results_data["bertscore_f1"] = bertscore_f1_list
        metrics["bertscore_f1"] = float(F1.mean().item())

    # COMET can also be slow, and it requires GPU for reasonable speed, so we make it optional and skip if there are no hypotheses.
    if comet_model is not None and hyps:
        comet_data = [{"src": s, "mt": mt, "ref": r} for s, mt, r in zip(sources, hyps, refs)]
        comet_out = comet_model.predict(
            comet_data,
            batch_size=8,
            gpus=1 if torch.cuda.is_available() else 0,
        )
        comet_scores_list = comet_out.scores  # per-example scores
        results_data["comet_score"] = comet_scores_list
        metrics["comet"] = float(comet_out.system_score)

    # Create and optionally save dataframe
    results_df = pd.DataFrame(results_data)
    if save_scores_path is not None:
        save_scores_path = Path(save_scores_path)
        save_scores_path.parent.mkdir(parents=True, exist_ok=True)
        results_df.to_csv(save_scores_path, index=False)
        print(f"Saved detailed evaluation results to {save_scores_path}")

    model.config.use_cache = prev_use_cache
    return metrics, sources, refs, hyps, results_df

In [ ]:
# -------------------------
# Train loop
# -------------------------
def train_one_run(
    train_ds,
    val_ds,
    tokenizer,
    lambda_embed=LAMBDA_EMBED,
    gamma_uniform=GAMMA_UNIFORM,
    embed_loss_type=EMBED_LOSS_TYPE,
    use_word_aware_loss=USE_WORD_AWARE_LOSS,
    use_uniform_loss=USE_UNIFORM_LOSS,
    embedding_source=EMBEDDING_SOURCE,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    batch_size=BATCH_SIZE,
    eval_generation_max_batches=EVAL_GENERATION_MAX_BATCHES,
    eval_generation_every=EVAL_GENERATION_EVERY,
    log_path=None,
):
    # initialize model, dataloaders, optimizer, scheduler
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.config.use_cache = False

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=make_collate_fn(tokenizer, use_word_aware_loss=use_word_aware_loss),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate_fn(tokenizer, use_word_aware_loss=use_word_aware_loss),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = num_epochs * len(train_loader)
    warmup_steps = max(1, int(0.1 * total_steps)) # 10% warmup
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    history = []
    best_state = None
    best_val_loss = float("inf")

    if log_path is None:
        tag = (
            f"lam{lambda_embed}_uni{gamma_uniform}_"
            f"word{int(use_word_aware_loss)}_u{int(use_uniform_loss)}_"
            f"emb{embedding_source}_{embed_loss_type}"
        )
        log_path = LOG_DIR / RUN_NAME / f"{RUN_NAME}_{tag}.jsonl"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    if log_path.exists():
        log_path.unlink()

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        model.train()

        train_sum = defaultdict(float)
        train_count = 0

        for batch in tqdm(train_loader, desc="Training"):
            batch = move_to_device(batch, DEVICE)

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            logits = outputs.logits

            # compute total loss with auxiliary embedding and uniformity terms
            # stats = {"loss": total_loss, "ce_loss": ce_loss, "embed_loss": embed_loss, "uniform_loss": uni_loss, "n_aux_vecs": n_aux_vecs}
            loss, stats = total_loss(
                logits=logits,
                labels=batch["labels"],
                token_embedding_weight=get_embedding_weight(model, embedding_source=embedding_source),
                tokenizer=tokenizer,
                pad_token_id=tokenizer.pad_token_id,
                lambda_embed=lambda_embed,
                gamma_uniform=gamma_uniform,
                embed_loss_type=embed_loss_type,
                use_word_aware_loss=use_word_aware_loss,
                use_uniform_loss=use_uniform_loss,
                word_spans=batch.get("word_spans"),
            )

            # backprop and step
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()

            # aggregate stats for the epoch
            bs = batch["labels"].size(0)
            train_count += bs
            # we multiply by batch size here because the loss is averaged over the batch, but we want to average over all examples at the end of the epoch
            for k, v in stats.items():
                train_sum[k] += float(v) * bs

        train_metrics = _aggregate_stats(train_sum, train_count)

        # Fast full validation loss every epoch
        val_loss_metrics = evaluate_loss(
            model,
            val_loader,
            tokenizer,
            lambda_embed=lambda_embed,
            gamma_uniform=gamma_uniform,
            embed_loss_type=embed_loss_type,
            use_word_aware_loss=use_word_aware_loss,
            use_uniform_loss=use_uniform_loss,
            embedding_source=embedding_source,
        )

        # Expensive generation metrics only on a capped subset
        gen_metrics = {}
        if ((epoch + 1) % eval_generation_every == 0) or (epoch == num_epochs - 1):
            gen_metrics, _, _, _, _ = evaluate_generation(
                model,
                val_loader,
                tokenizer,
                use_bertscore=False,
                comet_model=None,
                max_batches=eval_generation_max_batches,
                save_scores_path=log_path.with_suffix(f".epoch{epoch+1}_gen_scores.csv"),
            )

        record = {
            "epoch": epoch + 1,
            "lambda_embed": lambda_embed,
            "gamma_uniform": gamma_uniform,
            "use_word_aware_loss": use_word_aware_loss,
            "use_uniform_loss": use_uniform_loss,
            "embedding_source": embedding_source,
            "embed_loss_type": embed_loss_type,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_loss_metrics.items()},
            **{f"gen_{k}": v for k, v in gen_metrics.items()},
        }

        history.append(record)
        append_jsonl(log_path, record)

        # best checkpoint by validation loss (always available)
        if val_loss_metrics["loss"] < best_val_loss:
            best_val_loss = val_loss_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"epoch {epoch+1}: "
            f"train_loss={train_metrics['loss']:.4f}, "
            f"val_loss={val_loss_metrics['loss']:.4f}, "
            f"val_ce={val_loss_metrics['ce_loss']:.4f}, "
            f"val_embed={val_loss_metrics['embed_loss']:.4f}, "
            f"val_uniform={val_loss_metrics['uniform_loss']:.4f}"
        )
        if gen_metrics:
            print(
                f"    gen_bleu={gen_metrics.get('bleu', 0.0):.2f}, "
                f"gen_chrfpp={gen_metrics.get('chrfpp', 0.0):.2f}, "
                f"n_eval_examples={gen_metrics.get('n_eval_examples', 0)}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, log_path # str(log_path)


def run_sweep(experiments=None):
    """
    Run a list of experiment dictionaries.

    Example:
        experiments = [
            dict(name="baseline", lambda_embed=0.0, use_word_aware_loss=False,
                 use_uniform_loss=False, embedding_source="input"),
            dict(name="word_aware", lambda_embed=0.1, use_word_aware_loss=True,
                 use_uniform_loss=True, embedding_source="input"),
            dict(name="output_emb", lambda_embed=0.1, use_word_aware_loss=True,
                 use_uniform_loss=True, embedding_source="output"),
        ]
    """
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    # split: 80% train, 10% val, 10% test
    train_hf, val_hf, test_hf = load_translation_data()

    train_ds = ParallelTextDataset(train_hf)
    val_ds = ParallelTextDataset(val_hf)
    test_ds = ParallelTextDataset(test_hf)

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=make_collate_fn(tokenizer, use_word_aware_loss=USE_WORD_AWARE_LOSS),
    )

    comet_model = None
    if COMET_AVAILABLE:
        try:
            comet_ckpt = download_model("Unbabel/wmt22-comet-da")
            comet_model = load_from_checkpoint(comet_ckpt)
        except Exception:
            comet_model = None

    if experiments is None:
        experiments = [
            dict(
                name="baseline",
                lambda_embed=0.0,
                gamma_uniform=0.0,
                use_word_aware_loss=False,
                use_uniform_loss=False,
                embedding_source="input",
                embed_loss_type=EMBED_LOSS_TYPE,
            ),
            dict(
                name="word_aware_input",
                lambda_embed=LAMBDA_EMBED,
                gamma_uniform=GAMMA_UNIFORM,
                use_word_aware_loss=True,
                use_uniform_loss=True,
                embedding_source="input",
                embed_loss_type=EMBED_LOSS_TYPE,
            ),
        ]

    results = []
    for exp in experiments:
        exp = dict(exp)
        name = exp.pop("name", "run")
        print(f"\\n=== {name} ===")

        model, history, log_path = train_one_run(
            train_ds=train_ds,
            val_ds=val_ds,
            tokenizer=tokenizer,
            num_epochs=NUM_EPOCHS,
            lr=LR,
            batch_size=BATCH_SIZE,
            eval_generation_max_batches=EVAL_GENERATION_MAX_BATCHES,
            eval_generation_every=EVAL_GENERATION_EVERY,
            log_path=LOG_DIR / RUN_NAME / f"{RUN_NAME}_{name}.jsonl",
            **exp,
        )

        test_metrics, sources, refs, hyps, test_scores_df = evaluate_generation(
            model,
            test_loader,
            tokenizer,
            use_bertscore=False,
            comet_model=comet_model,
            max_batches=None,
            save_scores_path=LOG_DIR / RUN_NAME / f"{RUN_NAME}_{name}_test_scores.csv",
        )

        row = {
            "name": name,
            **exp,
            **test_metrics,
            "history": history,
            "log_path": log_path,
            "test_scores_df": test_scores_df,
        }
        results.append(row)

        print("test metrics:", {k: v for k, v in test_metrics.items() if isinstance(v, (int, float))})
        # save results after each run in case of interruption
        with open(LOG_DIR / RUN_NAME / f"{RUN_NAME}_results.json", "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2, default=str)

    return results

lambda = 1.0, 0.1, 0.5, 2.0



In [32]:
experiments = []
for lambda_val in [10.0]: # 0.0 # 0.1, 1.0
    experiments.append(dict(
            name=f"baseline_lambda_{lambda_val}",
            lambda_embed=lambda_val,
            gamma_uniform=0.0,
            use_word_aware_loss=False,
            use_uniform_loss=False,
            embedding_source="input",
            embed_loss_type=EMBED_LOSS_TYPE,
        ))
#     dict(
#         name="word_aware_input",
#         lambda_embed=LAMBDA_EMBED,
#         gamma_uniform=GAMMA_UNIFORM,
#         use_word_aware_loss=False,
#         use_uniform_loss=False,
#         embedding_source="input",
#         embed_loss_type=EMBED_LOSS_TYPE,
#     ),
#     dict(
#         name="word_aware_output",
#         lambda_embed=LAMBDA_EMBED,
#         gamma_uniform=GAMMA_UNIFORM,
#         use_word_aware_loss=True,
#         use_uniform_loss=False,
#         embedding_source="output",
#         embed_loss_type=EMBED_LOSS_TYPE,
#     ),
# ]

# results = run_sweep(experiments=experiments)
# # with open("results.json", "w") as f:
# #     json.dump(results, f, indent=2)

In [33]:
experiments

[{'name': 'baseline_lambda_10.0',
  'lambda_embed': 10.0,
  'gamma_uniform': 0.0,
  'use_word_aware_loss': False,
  'use_uniform_loss': False,
  'embedding_source': 'input',
  'embed_loss_type': 'cosine'}]

In [34]:
results = run_sweep(experiments=experiments)
# with open("results.json", "w") as f:
#     json.dump(results, f, indent=2)

HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-es-en/c96e2c5399ebfae4fc43d9669556b9afa74bb69d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/Helsinki-NLP/opus_books/resolve/main/RE

\n=== baseline_lambda_10.0 ===


HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/commits/main "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 28351.03it/s]
HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/discussions?p=0 "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-es-en/commits/refs%2Fpr%2F6 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/refs%2Fpr%2F6/model.safetensors.index.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-es-en/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/op

Epoch 1/3


Eval loss: 100%|██████████| 585/585 [00:14<00:00, 39.13it/s]
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to

...

epoch 2: train_loss=2.8564, val_loss=2.8143, val_ce=2.0144, val_embed=0.0800, val_uniform=0.0000
    gen_bleu=22.60, gen_chrfpp=43.33, n_eval_examples=64
Epoch 3/3


Eval loss: 100%|██████████| 585/585 [00:14<00:00, 39.17it/s]
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to

epoch 3: train_loss=2.5359, val_loss=2.7471, val_ce=1.9945, val_embed=0.0753, val_uniform=0.0000
    gen_bleu=23.31, gen_chrfpp=43.89, n_eval_examples=64


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-es-en")

for text in ["big", "large", "He was sad", "He was unhappy"]:
    print(text, "->", tok.tokenize(text))

big -> ['▁', 'big']
large -> ['▁la', 'rge']
He was sad -> ['▁He', '▁wa', 's', '▁sa', 'd']
He was unhappy -> ['▁He', '▁wa', 's', '▁un', 'ha', 'ppy']
